# Kaggle - Brain Tumor MRI Dataset

You can find the dataset and some informations about on the [Kaggle page](https://www.kaggle.com/datasets/masoudnickparvar/brain-tumor-mri-dataset).

For details on steps below, please see documentation in the *docs* directory.

## Google Collab setup
### Installations

In [6]:
!pip install mlflow

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.1/40.1 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 63.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 80.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 50.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.0/85.0 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 780.5/780.5 kB 47.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.3/207.3 kB 11.0 MB/s eta 0:00:00


### Data location

In [7]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [8]:
import os
os.listdir('/content/drive/MyDrive/data/Brain Tumor MRI')

['data', 'models']

## General

In [9]:
import mlflow
mlflow.set_tracking_uri("https://preachingly-nonabjuratory-marget.ngrok-free.dev")
mlflow.set_experiment("keras_example")

<Experiment: artifact_location='mlflow-artifacts:/470074894443755855', creation_time=1764782216982, experiment_id='470074894443755855', last_update_time=1764782216982, lifecycle_stage='active', name='keras_example', tags={}>

In [10]:
import numpy as np

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import DenseNet121
from tensorflow.keras.callbacks import ReduceLROnPlateau, EarlyStopping

import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path

In [11]:
# path management
PROJECT_ROOT = '/content/drive/MyDrive/data/Brain Tumor MRI'
PREP_DIR = PROJECT_ROOT + "/data/processed"
ARTEFACTS_DIR = PROJECT_ROOT + "/models"

CLASSES = ["notumor", "glioma", "meningioma", "pituitary"]

# parameters
IMG_SIZE = 260
SEED = 42

## Modeling

### Backbone

In [12]:
# 1. Create DenseNet121 WITHOUT weights
backbone = DenseNet121(
    include_top=False,
    weights=None,
    input_shape=(IMG_SIZE, IMG_SIZE, 3)
)

# 2. Load RadImageNet weights
backbone.load_weights(ARTEFACTS_DIR + "/RadImageNet-DenseNet121_notop.h5")

# 3. Freeze the backbone for firsts training
backbone.trainable = False

print("✅ RadImageNet DenseNet121 loaded successfully")

✅ RadImageNet DenseNet121 loaded successfully


In [13]:
#backbone.summary()

In [14]:
w = backbone.weights[0].numpy()
print("Mean:", np.mean(w), "Std:", np.std(w))

Mean: -0.0028284893 Std: 0.10494729


Model seems to be correctly loaded.

### Model definition

In [15]:
model_shared_part = keras.Sequential([
    #Data augmentation
    #TODO if need
    # Base
    backbone,
    # Head
    layers.GlobalAveragePooling2D(), # to flatten backbone output but with moderate position importance and more stable for MRI
    layers.Dense(512, use_bias=False), # 512 because it half of the backbone output (1024)
    layers.BatchNormalization(), # to normalize weight before heads
    layers.Activation('relu'),
    #layers.Dropout(0.4) # to reduce over-fitting risks
], name='shared_part')

In [16]:
model_head1 = keras.Sequential([
    layers.Dense(128, use_bias=False), # TO TEST : 256 ?
    layers.BatchNormalization(),
    layers.Activation('relu'),
    #layers.Dropout(0.2),
    layers.Dense(1,activation='sigmoid')
], name='tumor_presence')

In [17]:
model_head2 = keras.Sequential([
    layers.Dense(128, use_bias=False), # TO TEST : 256 ?
    layers.BatchNormalization(),
    layers.Activation('relu'),
    #layers.Dropout(0.2),
    layers.Dense(4,activation='softmax')
], name='tumor_type')

In [18]:
inputs = keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
x = model_shared_part(inputs)
output1 = model_head1(x)
output2 = model_head2(x)

model = keras.Model(
    inputs=inputs,
    outputs={
        "tumor_presence": output1,
        "tumor_type": output2
    },
    name='densenet_two_head'
)

In [19]:
loss_presence = keras.losses.BinaryFocalCrossentropy(
    gamma=2.0,
    alpha=0.25 # to favorize tumor detection, but taking account that tumors are 75% of data
)

In [20]:
loss_type = keras.losses.SparseCategoricalCrossentropy(from_logits=False)

In [21]:
model.compile(
    optimizer=keras.optimizers.Adam(), # change learning_rate for 1e-4 in fine-tuning steps
    loss={
        "tumor_presence": loss_presence,
        "tumor_type": loss_type,
    },

    metrics={
        "tumor_presence": [
            keras.metrics.BinaryAccuracy(name="accuracy"),
            keras.metrics.Recall(name="recall"),
            keras.metrics.Precision(name="precision"),
            keras.metrics.AUC(name="auc")
        ],
        "tumor_type": ["accuracy"],
    }
)

In [22]:
#model.summary()

## Streaming Training

In [23]:
def parse_tfrecord(example_proto):
    """
    Parse a single TFRecord example and convert grayscale → RGB.
    """
    feature_description = {
        "image": tf.io.FixedLenFeature([], tf.string),
        "label": tf.io.FixedLenFeature([], tf.int64),
    }

    example = tf.io.parse_single_example(example_proto, feature_description)

    # Deserialize image
    image = tf.io.parse_tensor(example["image"], out_type=tf.float32)

    # Shape after loading: (260, 260, 3)
    image.set_shape((260, 260, 3))

    label = tf.cast(example["label"], tf.int32)

    return image, label



def load_tfrecord_dataset(tfrecord_dir, shuffle=False, batch_size=1, repeat=False):
    """
    Load a TFRecord dataset from a directory.

    Args:
        tfrecord_dir (str or Path): Folder containing .tfrecord files
        shuffle (bool): Whether to shuffle files and samples
        batch_size (int): Batch size (can stay 1)
        repeat (bool): Repeat dataset indefinitely (for training)

    Returns:
        tf.data.Dataset
    """
    tfrecord_files = tf.io.gfile.glob(
        str(tfrecord_dir) + "/*.tfrecord"
    )

    ds = tf.data.TFRecordDataset(
        tfrecord_files,
        num_parallel_reads=tf.data.AUTOTUNE
    )

    ds = ds.map(
        parse_tfrecord,
        num_parallel_calls=tf.data.AUTOTUNE
    )

    if shuffle:
        ds = ds.shuffle(buffer_size=512)

    if repeat:
        ds = ds.repeat()

    ds = ds.batch(batch_size)
    ds = ds.prefetch(tf.data.AUTOTUNE)

    return ds


In [54]:
TRAIN_DIR = PREP_DIR + "/Training"
VAL_DIR   = PREP_DIR + "/Validation"
TEST_DIR  = PREP_DIR + "/Testing"

train_ds = load_tfrecord_dataset(
    TRAIN_DIR,
    shuffle=True,
    batch_size=1,
    repeat=False
)

val_ds = load_tfrecord_dataset(
    VAL_DIR,
    shuffle=False,
    batch_size=1,
    repeat=False
)
"""
test_ds = load_tfrecord_dataset(
    TEST_DIR,
    shuffle=False,
    batch_size=1,
    repeat=False
)
"""

'\ntest_ds = load_tfrecord_dataset(\n    TEST_DIR,\n    shuffle=False,\n    batch_size=1,\n    repeat=False\n)\n'

In [46]:
def split_labels(image, label):
    """
    Create labels for a 2-head model.
    """
    tumor_present = tf.cast(label != 0, tf.float32)
    tumor_type = tf.cast(label, tf.int32)

    return image, {
        "tumor_presence": tumor_present,
        "tumor_type": tumor_type
    }, { #sample_weight
        "tumor_presence": tf.ones_like(tumor_present), # always 1
        "tumor_type": tumor_present # mask if no tumor
    }

In [55]:
train_ds = train_ds.map(split_labels, num_parallel_calls=tf.data.AUTOTUNE)
val_ds = val_ds.map(split_labels, num_parallel_calls=tf.data.AUTOTUNE)

In [37]:
for x, y, z in train_ds.take(1):
    print("Image:")
    print(x.dtype, x.shape)
    print("\nValues:")
    for k, v in y.items():
        print(k, v.dtype, v.shape)
    print("\nWeight:")
    for k, v in z.items():
        print(k, v.dtype, v.shape)

Image:
<dtype: 'float32'> (1, 260, 260, 3)

Values:
tumor_presence <dtype: 'float32'> (1,)
tumor_type <dtype: 'int32'> (1,)

Weight:
tumor_type <dtype: 'float32'> (1,)


In [30]:
def count_tfrecord_samples(directory_path):
    """
    Count the number of TFRecord files in a directory.
    Assumes 1 sample per TFRecord.
    """
    directory_path = Path(directory_path)
    return len(list(directory_path.glob("*.tfrecord")))

In [31]:
def debugg_ds_size(train_ds, val_ds):
  print(">>>")
  print("Train cardinality:", tf.data.experimental.cardinality(train_ds).numpy())
  print("Val cardinality:", tf.data.experimental.cardinality(val_ds).numpy())

  print("Testing train_ds iteration")
  for batch in train_ds.take(1):
      print("Train batch OK")
      break
  else:
      print("❌ train_ds is EMPTY")

  print("Testing val_ds iteration")
  for batch in val_ds.take(1):
      print("Val batch OK")
      break
  else:
      print("❌ val_ds is EMPTY")

  print("<<<\n")

In [48]:
print(train_ds.element_spec)

(TensorSpec(shape=(None, 260, 260, 3), dtype=tf.float32, name=None), {'tumor_presence': TensorSpec(shape=(None,), dtype=tf.float32, name=None), 'tumor_type': TensorSpec(shape=(None,), dtype=tf.int32, name=None)}, {'tumor_presence': TensorSpec(shape=(None,), dtype=tf.float32, name=None), 'tumor_type': TensorSpec(shape=(None,), dtype=tf.float32, name=None)})


In [58]:
reduce_lr = ReduceLROnPlateau(
    monitor="val_tumor_presence_recall",
    mode="max",
    factor=0.5,
    patience=5,
    min_lr=1e-6,
    verbose=1
)

early_stopping = EarlyStopping(
    monitor="val_tumor_presence_recall",
    mode="max",
    min_delta=0.001,
    patience=10,
    restore_best_weights=True,
    verbose=1,
)

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=50,
    #steps_per_epoch=count_tfrecord_samples(TRAIN_DIR),
    #validation_steps=count_tfrecord_samples(VAL_DIR),
    callbacks=[reduce_lr, early_stopping],
)

Epoch 1/50


ValueError: Attr 'Toutput_types' of 'OptionalFromValue' Op passed list of length 0 less than minimum 1.

In [ ]:
history_df = pd.DataFrame(history.history)
history_df.loc[:, ['loss', 'val_loss']].plot();
print("Minimum validation loss: {}".format(history_df['val_loss'].min()))

In [ ]:
Warning : do not forget :
- integerer mlflow -> premier essai avec visu
- activer dropoutS dans model
- stratégie de fine-tuning du backbone
- BatchNormalization precaution in fine-tuning
- maximize reccal (y_pred > 0.3 → tumeur)